# GRAM Recurrent Qwen - Phase 3 Resume

Clean resume notebook for the current successful state:

- Phase 1 deterministic halting baseline: `outputs/qwen_0_5b_phase1_a100_beta008_continue_150/phase1_step_150.pt`
- Phase 2 stochastic post-latent scale-5 checkpoint: `outputs/qwen_0_5b_phase2_post_latent_scale5_steps25/phase2_step_25.pt`

Current result: stochastic trajectories now produce measurable hidden-state diversity and short-answer candidate diversity. Next step is evaluation/reranking, not more training.

## 0. Runtime, Dependencies, And HF Auth

Use a GPU runtime. This notebook is designed to resume without wiping `/content/gram-recurrent-qwen/data` or `/content/gram-recurrent-qwen/outputs`.

In [ ]:
import os, sys, subprocess, json, textwrap
from pathlib import Path

print('python', sys.version)
try:
    import torch
    print('torch', torch.__version__)
    print('cuda available', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu', torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch import before install failed:', repr(exc))

subprocess.run(['nvidia-smi'], check=False)

In [ ]:
!pip -q install -U "transformers>=4.44" accelerate pyyaml pytest sentencepiece safetensors datasets huggingface_hub

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

token = userdata.get('HF_TOKEN')
assert token, 'HF_TOKEN missing from Colab Secrets. Add it in the key sidebar and enable notebook access.'
os.environ['HF_TOKEN'] = token
os.environ['HUGGINGFACE_HUB_TOKEN'] = token
login(token=token, add_to_git_credential=False)
who = HfApi(token=token).whoami()
print('HF auth OK:', who.get('name') or who.get('email') or 'authenticated user')

## 1. Patch Or Load Project Code Without Deleting Outputs

If `/content/gram-recurrent-qwen` already has the patched code, this cell does nothing. If the project is missing or stale, upload the current `gram-recurrent-qwen-colab-upload.zip`; the cell replaces code/config folders but preserves `data/` and `outputs/`.

In [ ]:
import os, shutil, zipfile
from pathlib import Path
from google.colab import files

PROJECT_ROOT = Path('/content/gram-recurrent-qwen')
PATCH_ROOT = Path('/content/gram-recurrent-qwen-patch')

def project_is_patched(root: Path) -> bool:
    wrapper = root / 'models' / 'recurrent_wrapper.py'
    traj = root / 'models' / 'trajectory_utils.py'
    if not wrapper.exists() or not traj.exists():
        return False
    return (
        'latent_injection_mode' in wrapper.read_text(encoding='utf-8')
        and 'pooled_by_trajectory.float()' in traj.read_text(encoding='utf-8')
    )

if PROJECT_ROOT.exists() and project_is_patched(PROJECT_ROOT):
    print('Project already present and patched:', PROJECT_ROOT)
else:
    print('Upload gram-recurrent-qwen-colab-upload.zip')
    if PATCH_ROOT.exists():
        shutil.rmtree(PATCH_ROOT)
    PATCH_ROOT.mkdir(parents=True)

    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    assert zip_names, 'Upload gram-recurrent-qwen-colab-upload.zip'

    with zipfile.ZipFile(zip_names[0]) as zf:
        zf.extractall(PATCH_ROOT)

    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for name in ['models', 'training', 'eval', 'tests', 'config', 'colab', 'runpod', 'scripts']:
        src = PATCH_ROOT / name
        dst = PROJECT_ROOT / name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)

    for name in ['README.md', 'requirements.txt', 'infer_recurrent.py', '.gitignore']:
        src = PATCH_ROOT / name
        if src.exists():
            shutil.copy2(src, PROJECT_ROOT / name)

os.chdir(PROJECT_ROOT)
os.environ['PYTHONPATH'] = str(PROJECT_ROOT)
print('project root:', PROJECT_ROOT)
!python -m pytest -q tests
assert project_is_patched(PROJECT_ROOT)
print('Patch checks OK')

## 2. Constants And Checkpoint Inventory

The next cells assume the current best checkpoints exist in `outputs/`. If they were wiped by a Colab restart, restore them from Drive in the following cell.

In [ ]:
%cd /content/gram-recurrent-qwen

import os, sys, subprocess, gc, re, math, json, random
from pathlib import Path
import torch

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
SPLIT = '6,18'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TRAIN_DTYPE = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else ('float16' if torch.cuda.is_available() else 'float32')
ADAPTER_DTYPE = 'float32'

PHASE1_CKPT = 'outputs/qwen_0_5b_phase1_a100_beta008_continue_150/phase1_step_150.pt'
PHASE2_SCALE2_CKPT = 'outputs/qwen_0_5b_phase2_post_latent_scale2_steps50/phase2_step_50.pt'
PHASE2_SCALE5_CKPT = 'outputs/qwen_0_5b_phase2_post_latent_scale5_steps25/phase2_step_25.pt'

print({
    'MODEL_NAME': MODEL_NAME,
    'SPLIT': SPLIT,
    'DEVICE': DEVICE,
    'TRAIN_DTYPE': TRAIN_DTYPE,
    'ADAPTER_DTYPE': ADAPTER_DTYPE,
})

print('\nCheckpoints:')
for p in sorted(Path('outputs').glob('**/*.pt')) if Path('outputs').exists() else []:
    print(' ', p, f'{p.stat().st_size/1e6:.1f} MB')

for required in [PHASE1_CKPT, PHASE2_SCALE5_CKPT]:
    print(required, 'OK' if Path(required).exists() else 'MISSING')

## 3. Optional: Restore Or Save Checkpoints To Drive

Run this if the required checkpoints are missing, or after a successful run to preserve them. The restore paths match the flat filenames created by the previous save cell.

In [ ]:
RESTORE_FROM_DRIVE = False
SAVE_TO_DRIVE = True

from pathlib import Path
from google.colab import drive
import shutil

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/gram-recurrent-qwen-checkpoints')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

restore_map = {
    'phase1_step_150.pt': Path(PHASE1_CKPT),
    'phase2_step_50.pt': Path(PHASE2_SCALE2_CKPT),
    'phase2_step_25.pt': Path(PHASE2_SCALE5_CKPT),
}

if RESTORE_FROM_DRIVE:
    for src_name, dst in restore_map.items():
        src = DRIVE_DIR / src_name
        if src.exists() and not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            print('restored', src, '->', dst)

if SAVE_TO_DRIVE:
    for ckpt in [Path(PHASE1_CKPT), Path(PHASE2_SCALE2_CKPT), Path(PHASE2_SCALE5_CKPT)]:
        if ckpt.exists():
            dst = DRIVE_DIR / ckpt.name
            shutil.copy2(ckpt, dst)
            print('saved', ckpt, '->', dst)
        else:
            print('missing, not saved:', ckpt)

## 4. Validate Current Best Checkpoints

This is a quick sanity gate. Expected current values are approximately:

- Phase 1: `expected_ce ~2.74`, `mean_expected_loops ~2.90`
- Phase 2 scale 5: `trajectory_diversity ~0.008`, `expected_ce ~2.72`, `mean_expected_loops ~2.87`

In [ ]:
import os, sys, subprocess
from pathlib import Path

env = os.environ.copy()

def run(cmd):
    print('\n$', ' '.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print('STDERR:')
        print(result.stderr)
    result.check_returncode()
    return result

assert Path(PHASE1_CKPT).exists(), PHASE1_CKPT
assert Path(PHASE2_SCALE5_CKPT).exists(), PHASE2_SCALE5_CKPT

print('Phase 1 baseline validation')
run([
    sys.executable, 'eval/eval_jsonl.py',
    '--model_name', MODEL_NAME,
    '--data_jsonl', 'data/opus47_val.jsonl',
    '--checkpoint', PHASE1_CKPT,
    '--split', SPLIT,
    '--max_loops', '4',
    '--max_length', '512',
    '--beta', '0.08',
    '--dtype', TRAIN_DTYPE,
    '--adapter_dtype', ADAPTER_DTYPE,
    '--device', DEVICE,
])

print('Phase 2 scale-5 validation')
run([
    sys.executable, 'eval/eval_jsonl.py',
    '--model_name', MODEL_NAME,
    '--data_jsonl', 'data/opus47_val.jsonl',
    '--checkpoint', PHASE2_SCALE5_CKPT,
    '--split', SPLIT,
    '--max_loops', '4',
    '--num_trajectories', '2',
    '--sample_latents',
    '--latent_injection_mode', 'post',
    '--max_length', '512',
    '--beta', '0.08',
    '--eta', '1e-4',
    '--rho', '1e-3',
    '--dtype', TRAIN_DTYPE,
    '--adapter_dtype', ADAPTER_DTYPE,
    '--device', DEVICE,
])

## 5. Load A Checkpoint For Candidate Generation

Loads the Phase 2 scale-5 checkpoint once and defines a minimal no-cache greedy multi-trajectory generation helper.

In [ ]:
%cd /content/gram-recurrent-qwen

import os, gc, torch, re, random, json
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from eval.eval_identity import model_load_kwargs, parse_split
from models.lora import apply_lora_to_recurrent_block
from models.recurrent_wrapper import RecurrentQwenForCausalLM
from training.checkpointing import load_trainable_checkpoint
from models.trajectory_utils import repeat_for_trajectories

CKPT = PHASE2_SCALE5_CKPT
NUM_TRAJ = 4
MAX_NEW_TOKENS = 64
MAX_CONTEXT = 512

torch.manual_seed(1234)
random.seed(1234)

if 'wrapper' in globals():
    del wrapper
if 'model' in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    **model_load_kwargs(TRAIN_DTYPE, 'default'),
).to(DEVICE)

wrapper = RecurrentQwenForCausalLM(model, layer_split=parse_split(SPLIT)).to(DEVICE)
apply_lora_to_recurrent_block(wrapper, rank=8, alpha=16, dropout=0.0, adapter_dtype=torch.float32)
wrapper.set_trainable_modules_dtype(torch.float32)
load_info = load_trainable_checkpoint(wrapper, CKPT)
wrapper.eval()

print('loaded checkpoint:', CKPT)
print('loaded_keys:', len(load_info['loaded_keys']))
print('latent_scale:', float(wrapper.latent_trajectory.adapter.latent_scale.detach().cpu()))
print('adapter_std:', float(wrapper.latent_trajectory.adapter.proj.weight.detach().float().std().cpu()))

def encode_chat_prompt(user_prompt):
    messages = [
        {'role': 'system', 'content': 'You are a helpful AI assistant. Give a concise answer.'},
        {'role': 'user', 'content': user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_CONTEXT).to(DEVICE)

def greedy_multi_trajectory_generate(user_prompt, num_traj=NUM_TRAJ, max_new_tokens=MAX_NEW_TOKENS, temperature=0.0):
    encoded = encode_chat_prompt(user_prompt)
    input_ids = repeat_for_trajectories(encoded['input_ids'], num_traj)
    attention_mask = repeat_for_trajectories(encoded['attention_mask'], num_traj)
    finished = torch.zeros(num_traj, dtype=torch.bool, device=DEVICE)
    generated = [[] for _ in range(num_traj)]
    diversity_trace = []

    with torch.no_grad():
        for step in range(max_new_tokens):
            if input_ids.shape[1] > MAX_CONTEXT:
                input_ids = input_ids[:, -MAX_CONTEXT:]
                attention_mask = attention_mask[:, -MAX_CONTEXT:]

            out = wrapper(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_loops=4,
                num_trajectories=1,
                sample_latents=True,
                latent_injection_mode='post',
                use_cache=False,
                return_dict=True,
            )
            logits = out.logits[:, -1, :].float()
            if temperature and temperature > 0:
                probs = torch.softmax(logits / temperature, dim=-1)
                next_ids = torch.multinomial(probs, num_samples=1).squeeze(-1)
            else:
                next_ids = logits.argmax(dim=-1)
            next_ids = torch.where(finished, torch.full_like(next_ids, tokenizer.eos_token_id), next_ids)

            diversity_trace.append({
                'step': step,
                'unique_next': len(set(next_ids.detach().cpu().tolist())),
                'mean_expected_loops': float(out.metrics['mean_expected_loops']),
            })
            for k, tok in enumerate(next_ids.detach().cpu().tolist()):
                if not finished[k]:
                    generated[k].append(tok)
            finished |= next_ids.eq(tokenizer.eos_token_id)
            input_ids = torch.cat([input_ids, next_ids[:, None]], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_ids[:, None])], dim=1)
            if bool(finished.all()):
                break

    answers = [tokenizer.decode(ids, skip_special_tokens=True).strip() for ids in generated]
    return answers, diversity_trace

## 6. Open-Ended Candidate Diversity Probe

This reproduces the successful Phase 3 mini-pass and checks that trajectories compound into different continuations.

In [ ]:
open_prompts = [
    'Find one valid 4-queens placement.',
    'Give one valid 5-letter word using the letters R, A, T, E, S.',
    'A maze has two valid exits. Describe one valid path in plain English.',
    'Solve: If a train travels 120 miles in 3 hours, what is its average speed?',
    'Name one strategy for solving a Sudoku puzzle.',
    'Give one possible reason a model might hallucinate an answer.',
]

for prompt in open_prompts:
    print('\n' + '=' * 100)
    print('PROMPT:', prompt)
    answers, trace = greedy_multi_trajectory_generate(prompt, num_traj=4, max_new_tokens=48, temperature=0.0)
    for i, answer in enumerate(answers):
        print(f'\n--- trajectory {i} ---')
        print(answer if answer else '[empty]')
    unique_next_steps = [x['unique_next'] for x in trace]
    print('\nSUMMARY:', {
        'unique_answers': f'{len(set(answers))}/4',
        'max_unique_next': max(unique_next_steps) if unique_next_steps else 0,
        'avg_unique_next': sum(unique_next_steps) / max(len(unique_next_steps), 1),
        'steps': len(unique_next_steps),
    })

## 7. Known-Answer Best-Of-K Mini Eval

This is the next real question: does K=4 produce at least one candidate that a simple task scorer can select? The scorer is intentionally crude and task-specific.

In [ ]:
known_tasks = [
    {
        'name': 'train speed',
        'prompt': 'Solve concisely: If a train travels 120 miles in 3 hours, what is its average speed?',
        'patterns': [r'\b40\b', r'40\s*(mph|miles per hour)'],
    },
    {
        'name': 'pharmacy tubs',
        'prompt': 'If a pharmacy has 20 tubs, needs 100 tubs total, and buys one quarter of the remaining tubs from a new vendor, how many tubs must it buy from its usual vendor?',
        'patterns': [r'\b60\b'],
    },
    {
        'name': 'arithmetic',
        'prompt': 'Compute 17 + 28. Give only the answer.',
        'patterns': [r'\b45\b'],
    },
    {
        'name': 'rates word',
        'prompt': 'Give one valid English 5-letter word using exactly the letters R, A, T, E, S.',
        'patterns': [r'\b(stare|rates|tears|tares|aster|resat)\b'],
    },
    {
        'name': 'sudoku strategy',
        'prompt': 'Name one standard strategy for solving a Sudoku puzzle. Give a concise answer.',
        'patterns': [r'backtracking|elimination|naked single|hidden single|candidate'],
    },
]

def task_hit(answer, patterns):
    text = answer.lower()
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)

results = []
for task in known_tasks:
    print('\n' + '=' * 100)
    print(task['name'].upper(), '-', task['prompt'])
    answers, trace = greedy_multi_trajectory_generate(task['prompt'], num_traj=4, max_new_tokens=64, temperature=0.0)
    hits = [task_hit(a, task['patterns']) for a in answers]
    for i, (answer, hit) in enumerate(zip(answers, hits)):
        print(f'\n--- trajectory {i} hit={hit} ---')
        print(answer if answer else '[empty]')
    summary = {
        'name': task['name'],
        'hits': sum(hits),
        'best_of_4': any(hits),
        'unique_answers': len(set(answers)),
        'avg_unique_next': sum(x['unique_next'] for x in trace) / max(len(trace), 1),
    }
    print('\nSUMMARY:', summary)
    results.append(summary)

print('\n' + '=' * 100)
print('OVERALL')
print('best_of_4_hits:', sum(r['best_of_4'] for r in results), '/', len(results))
print('total_candidate_hits:', sum(r['hits'] for r in results), '/', 4 * len(results))
print(json.dumps(results, indent=2))

## 8. Save Checkpoints And Notes

Run after useful results. This preserves the current best checkpoints plus generated notebook outputs in Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, os

drive.mount('/content/drive')
target = Path('/content/drive/MyDrive/gram-recurrent-qwen-checkpoints')
target.mkdir(parents=True, exist_ok=True)

for ckpt in [PHASE1_CKPT, PHASE2_SCALE2_CKPT, PHASE2_SCALE5_CKPT]:
    p = Path(ckpt)
    if p.exists():
        dst = target / p.name
        shutil.copy2(p, dst)
        print('saved', p, '->', dst)
    else:
        print('missing:', p)

print('checkpoint dir:')
!ls -lh /content/drive/MyDrive/gram-recurrent-qwen-checkpoints